In [1]:
import pandas as pd

from phd_project.config import config


cfg = config.load_config()

Combines the lists of records to download together into a single csv file for each database

In [2]:
import csv

# paths to record lists:
esm_records_AvgSA03_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_summary_esm_records_selected.csv"
ngasub_records_AvgSA03_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_summary_ngasub_records_selected.csv"
esm_records_AvgSA06_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_summary_esm_records_selected.csv"
ngasub_records_AvgSA06_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_summary_ngasub_records_selected.csv"

esm_combined_fp = cfg["proc_data"]["gm_selection"] / f"all_esm_records_to_download.csv"
ngasub_combined_fp = cfg["proc_data"]["gm_selection"] / f"all_ngasub_records_to_download.csv"
ngasub_rsn_batches_fp = cfg["proc_data"]["gm_selection"] / f"ngasub_rsns_to_download_batched.csv"

# --- ESM: combine the selected records from both campaigns ---
esm_06 = pd.read_csv(esm_records_AvgSA06_fp, header=[0, 1])
esm_03 = pd.read_csv(esm_records_AvgSA03_fp, header=[0, 1])
esm_selected = pd.concat([esm_06, esm_03], ignore_index=True).drop_duplicates()

# already-downloaded ESM records, from the hdf5 folder.
# filenames look like:  {event_id}__{station_code}__{location_code}.h5
downloaded_esm = set()
for f in cfg["raw_data"]["esm_hdf5_folder"].glob("*.h5"):
    parts = f.stem.split("__")
    if len(parts) == 3:
        ev, st, loc = parts
        downloaded_esm.add((ev, st, int(loc)))  # normalise "00" -> 0

# keep only selected records that are not yet downloaded
esm_keys = list(zip(
    esm_selected[("metadata", "event_id")],
    esm_selected[("metadata", "station_code")],
    esm_selected[("metadata", "location_code")].astype(int),
))
esm_to_download = esm_selected[[k not in downloaded_esm for k in esm_keys]].reset_index(drop=True)
esm_to_download.to_csv(esm_combined_fp, index=False)

# --- NGAsub: combine the selected records from both campaigns ---
nga_06 = pd.read_csv(ngasub_records_AvgSA06_fp, header=0)
nga_03 = pd.read_csv(ngasub_records_AvgSA03_fp, header=0)
nga_selected = pd.concat([nga_06, nga_03])
nga_selected = nga_selected.groupby("NGAsubRSN")["count"].sum().reset_index().sort_values("count", ascending=False)

# already-downloaded NGAsub records, from the folder (subdirs named NGASub_RSN_<rsn>)
downloaded_ngasub = {
    int(d.name.removeprefix("NGASub_RSN_"))
    for d in cfg["raw_data"]["ngasub_folder"].iterdir()
    if d.name.startswith("NGASub_RSN_")
}

nga_to_download = nga_selected[~nga_selected["NGAsubRSN"].isin(downloaded_ngasub)].reset_index(drop=True)
nga_to_download.to_csv(ngasub_combined_fp, index=False)

# batched RSN file: 30 RSNs per row, first cell = 1-indexed row number, no header
rsns = nga_to_download["NGAsubRSN"].tolist()
rsn_rows = [[i // 30 + 1, *rsns[i:i + 30]] for i in range(0, len(rsns), 30)]
with open(ngasub_rsn_batches_fp, "w", newline="") as fh:
    csv.writer(fh).writerows(rsn_rows)

# --- summary ---
print("Records still to download (selected minus already downloaded):")
print(f"  ESM:    {len(esm_to_download):>5} / {len(esm_selected)} selected  "
      f"({len(esm_selected) - len(esm_to_download)} already downloaded)")
print(f"  NGAsub: {len(nga_to_download):>5} / {len(nga_selected)} selected  "
      f"({len(nga_selected) - len(nga_to_download)} already downloaded)")
print(f"  NGAsub batched RSN file: {ngasub_rsn_batches_fp.name}  "
      f"({len(rsn_rows)} rows of up to 30)")

Records still to download (selected minus already downloaded):
  ESM:        0 / 1358 selected  (1358 already downloaded)
  NGAsub:   648 / 1813 selected  (1165 already downloaded)
  NGAsub batched RSN file: ngasub_rsns_to_download_batched.csv  (22 rows of up to 30)


In [3]:
# paths to per-campaign conversion lists:
convert_AvgSA03_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_03_records_to_convert.csv"
convert_AvgSA06_fp = cfg["proc_data"]["gm_selection"] / f"AvgSA_06_records_to_convert.csv"

esm_convert_fp = cfg["proc_data"]["gm_selection"] / f"esm_records_to_convert.csv"
ngasub_convert_fp = cfg["proc_data"]["gm_selection"] / f"ngasub_records_to_convert.csv"

conv_03 = pd.read_csv(convert_AvgSA03_fp, dtype=str)
conv_06 = pd.read_csv(convert_AvgSA06_fp, dtype=str)
conv_combined = pd.concat([conv_03, conv_06], ignore_index=True).drop_duplicates()

for db_name, fp in [("ESM", esm_convert_fp), ("NGASub", ngasub_convert_fp)]:
    sub = (conv_combined[conv_combined["database"] == db_name]
           [["record_identifier", "component"]]
           .drop_duplicates()
           .reset_index(drop=True))
    sub.to_csv(fp, index=False)
    print(f"{fp.name}: {len(sub)} records")


esm_records_to_convert.csv: 1848 records
ngasub_records_to_convert.csv: 2136 records
